In [1]:
# 1. 清理Colab預設可能衝突的舊版套件
!apt-get remove chromium-browser chromium-chromedriver -y
!apt-get autoremove -y

# 2. 下載並安裝官方Google Chrome穩定版
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb
!rm google-chrome-stable_current_amd64.deb

# 3. 安裝Linux虛擬螢幕套件
!apt-get install -y xvfb
!apt-get install -y nodejs
!apt-get install -y ffmpeg

# 4. 一次裝齊所有需要的Python爬蟲套件
!pip install undetected-chromedriver pyvirtualdisplay selenium beautifulsoup4 requests pandas yt-dlp
!pip install --upgrade yt-dlp
!pip install -U --pre "yt-dlp[default]"

!pip install easyocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
--2026-05-12 13:34:18--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 74.125.137.91, 74.125.137.93, 74.125.137.190, ...
Connecting to dl.google.com (dl.google.com)|74.125.137.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130210724 (124M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 124.18M   462MB/s    in 0.3s    

2026-05-12 13:34:19 (462 MB/s) - ‘google-chro

In [2]:
#連線測試
import undetected_chromedriver as uc
from pyvirtualdisplay import Display
import time

# 1. 啟動虛擬螢幕
print("正在架設虛擬顯示器...")
display = Display(visible=0, size=(1920, 1080))
display.start()

# 2. 設定啟動參數
options = uc.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

options.binary_location = "/usr/bin/google-chrome"

print("啟動Chrome...")
driver = uc.Chrome(options=options, version_main=148)

print("嘗試連線至嘖嘖...")
driver.get("https://www.zeczec.com/")

# 給Cloudflare更充裕的時間跑完JavaScript
time.sleep(8)

print("目前網頁標題：", driver.title)

# 測試完關閉資源
driver.quit()
display.stop()

正在架設虛擬顯示器...
啟動Chrome...
嘗試連線至嘖嘖...
目前網頁標題： 嘖嘖 zeczec × 讓美好的事物發生：台灣的群眾集資 (Crowdfunding) 平台


In [6]:
#爬蟲跑到一半當機，或手動按了停止鈕後，用來釋放RAM的
!pkill -f chrome

In [3]:
import os
import re
import time
import csv
import requests
import pandas as pd
import yt_dlp
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from pyvirtualdisplay import Display
from selenium.webdriver.common.by import By
from google.colab import drive
import shutil
import easyocr
from PIL import Image
import numpy as np

# ==========================================
# 新增功能區：1. 專案自動編號  2. FAQ 收集器
# ==========================================
#1.自動編號
category_prefix_map = {
    "出版": "P", "時尚": "F", "設計": "D", "科技": "T",
    "教育": "E", "遊戲": "G", "飲食": "FD", "社會": "S"
}
status_prefix_map = {"成功": "S", "失敗": "F"}
project_counters = {}

def generate_project_code(category_name, status):
    """產生如 GF1 (遊戲+失敗+第1號) 的編號"""
    cat_p = category_prefix_map.get(category_name, "X")
    stat_p = status_prefix_map.get(status, "U")
    prefix = f"{cat_p}{stat_p}"

    if prefix not in project_counters:
        project_counters[prefix] = 1
    else:
        project_counters[prefix] += 1
    return f"{prefix}{project_counters[prefix]}"

# 2.FAQ收集
def process_faq_data(driver, project_url, proj_code, save_folder_path, funding_days=30):
    """抓取 FAQ 並回傳 (總數, 更新頻率)"""
    try:
        # 1. 進入 FAQ 專屬網址 (嘖嘖支援直接用網址切換標籤)
        driver.get(f"{project_url}/faqs")
        time.sleep(3) # 【關鍵】必須等待標籤切換與動態框框渲染完畢


        total_count = 0
        tabs = driver.find_elements(By.XPATH, "//*[contains(text(), '常見問答')]")
        for tab in tabs:
            # 用正則表達式把數字抓出來
            match = re.search(r'常見問答\s*(\d+)', tab.text)
            if match:
                total_count = int(match.group(1))
                break

        # 算出頻率
        update_freq = round(total_count / funding_days, 4) if funding_days > 0 else 0

        # 3. 抓取底下的問答文字
        if total_count > 0:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            faq_text_content = f"專案編號：{proj_code}\nFAQ 總體數：{total_count}\n更新頻率：{update_freq}\n" + "="*30 + "\n\n"

            # 利用你截圖中的第二個特徵：每個問答都會有「更新於 YYYY/MM/DD」
            # 我們找到這些時間標籤，然後往上找它的父容器(也就是那個灰底的框框)
            date_texts = soup.find_all(string=re.compile(r'更新於\s*\d{4}'))

            if date_texts:
                for i, date_text in enumerate(date_texts, 1):
                    current_node = date_text.parent
                    while current_node.parent and len(current_node.parent.find_all(string=re.compile(r'更新於\s*\d{4}'))) == 1:
                        current_node = current_node.parent

                    if current_node:
                        text = current_node.get_text(separator='\n', strip=True)
                        faq_text_content += f"【第 {i} 題】\n{text}\n\n"

            else:
                # 備用方案：如果找不到「更新於」，就抓取整個網頁的主要內容文字
                faq_text_content += soup.get_text(separator='\n', strip=True)[:2000] # 只抓前2000字避免過長

            # 存成 TXT
            txt_file_path = os.path.join(save_folder_path, f"{proj_code}_FAQ.txt")
            with open(txt_file_path, "w", encoding="utf-8") as f:
                f.write(faq_text_content)

        return total_count, update_freq
    except Exception as e:
        print(f"FAQ抓取失敗: {e}")
        return 0, 0

# ==========================================
# 0. 系統環境修正 (解決 No supported JavaScript runtime)
# ==========================================
# 在 Colab 執行時，請先跑這行： !apt-get install -y nodejs

# ==========================================
# 1. 核心功能：多媒體下載器 (你的邏輯)
# ==========================================
def zeczec_multimodal_downloader(url, project_name, driver, base_folder_path, proj_code=""):
    # 建立圖片與影片子目錄 (這裡加上了 proj_code 前綴，例如 GF1_images)
    img_path = os.path.join(base_folder_path, f"{proj_code}_images")
    video_path = os.path.join(base_folder_path, f"{proj_code}_videos")
    os.makedirs(img_path, exist_ok=True)
    os.makedirs(video_path, exist_ok=True)

    print(f" [影音任務] 正在掃描：{project_name}")

    driver.get(url)
    time.sleep(3)

    # 模擬捲動觸發 Lazy Load
    for i in range(3):
        driver.execute_script(f"window.scrollTo(0, {i * 1200});")
        time.sleep(1.2)

    resource_log = []

    # --- [A. 圖片抓取] ---
    print("正在掃描圖片...")
    img_tasks = []
    for img in driver.find_elements(By.TAG_NAME, "img"):
        f_url = img.get_attribute("data-src") or img.get_attribute("src")
        if f_url: img_tasks.append(f_url)

    for div in driver.find_elements(By.CSS_SELECTOR, "div[style*='background-image']"):
        style = div.get_attribute("style")
        match = re.search(r'url\("?(.+?)"?\)', style)
        if match: img_tasks.append(match.group(1).replace('"', '').replace("'", ""))

    img_count = 0
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'}
    for img_url in set(img_tasks):
        if img_url.startswith("data:") or any(x in img_url.lower() for x in ['icon', 'logo', 'avatar', '.svg']):
            continue
        try:
            resp = requests.get(img_url, headers=headers, timeout=10)
            if resp.status_code == 200:
                ctype = resp.headers.get('Content-Type', '').lower()
                ext = ".gif" if 'gif' in ctype else ".png" if 'png' in ctype else ".webp" if 'webp' in ctype else ".jpg"
                f_path = os.path.join(img_path, f"img_{img_count}{ext}")
                with open(f_path, 'wb') as f: f.write(resp.content)
                if os.path.getsize(f_path) > 5000:
                    resource_log.append({"FileName": f"img_{img_count}{ext}", "Type": "Image/GIF", "URL": img_url})
                    img_count += 1
                else: os.remove(f_path)
        except: continue
    print(f"圖片下載完成：共 {img_count} 張")

    # --- [B. 影片偵測 - noscript 專攻版] ---
    video_urls = set()
    # 1. 掃描 iframe
    try:
        for iframe in driver.find_elements(By.TAG_NAME, "iframe"):
            v_src = iframe.get_attribute("src") or iframe.get_attribute("data-src")
            if v_src and "youtube.com" in v_src:
                video_urls.add(v_src.split('?')[0].replace("embed/", "watch?v="))
    except: pass

    # 2. 掃描 noscript
    try:
        for ns in driver.find_elements(By.TAG_NAME, "noscript"):
            ns_content = ns.get_attribute("innerHTML")
            if "youtube.com" in ns_content and "embed" in ns_content:
                match = re.search(r'embed/([a-zA-Z0-9_-]{11})', ns_content)
                if match:
                    video_urls.add(f"https://www.youtube.com/watch?v={match.group(1)}")
    except: pass

    # --- [C. 影片下載] ---
    if video_urls:
        ydl_opts = {
            'format': 'best',
            'outtmpl': f'{video_path}/video_%(title)s.%(ext)s',
            'quiet': True,
            'no_warnings': True,
            'ignoreerrors': True,
            'sleep_interval': 5,
            'max_sleep_interval': 12
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            for v_url in video_urls:
                try:
                    print(f"下載影片: {v_url}")
                    ydl.download([v_url])
                    resource_log.append({"FileName": v_url, "Type": "Video", "URL": v_url})
                except: print(f"影片下載失敗，可能被 YouTube 封鎖")

    # 儲存下載紀錄 (這裡加上了 proj_code 前綴，例如 GF1_resource_log.csv)
    if resource_log:
        pd.DataFrame(resource_log).to_csv(f"{base_folder_path}/{proj_code}_resource_log.csv", index=False, encoding="utf-8-sig")

# ==========================================
# 2. 環境與初始化設定
# ==========================================
MAIN_TYPES = {'群眾募資': '0', '預購式專案': '1'}
SUB_CATEGORIES = {
    #'出版': '3',
    #'時尚': '7',
    #'設計': '8',
    #'科技': '11',
    '教育': '12',
    '遊戲': '13',
    #'飲食': '14',
    #'社會': '15'
}

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data'
#BASE_DIR = './ZecZec_Dataset'

def create_directories():
    if not os.path.exists(BASE_DIR): os.mkdir(BASE_DIR)
    for main_name in MAIN_TYPES.keys():
        main_path = os.path.join(BASE_DIR, main_name)
        if not os.path.exists(main_path): os.mkdir(main_path)
        for sub_name in SUB_CATEGORIES.keys():
            sub_path = os.path.join(main_path, sub_name)
            if not os.path.exists(sub_path): os.mkdir(sub_path)
            for status_name in ['成功', '失敗']:
                status_path = os.path.join(sub_path, status_name)
                if not os.path.exists(status_path): os.mkdir(status_path)
    print("資料夾結構建立完成！")

def get_driver():
    print("啟動Chrome...")
    display = Display(visible=0, size=(1920, 1080))
    display.start()
    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.binary_location = "/usr/bin/google-chrome"
    driver = uc.Chrome(options=options, version_main=148)
    return driver, display

# ==========================================
# 3. 核心解析邏輯 (文字抓取)
# ==========================================
def scrape_project_details(driver, project_url, main_name, sub_name):
    driver.get(project_url)
    time.sleep(3)

    # 18歲驗證
    try:
        xpath_query = "//*[self::button or self::a or self::input][contains(., '我已滿 18 歲')]"
        age_btn = driver.find_element(By.XPATH, xpath_query)
        driver.execute_script("arguments[0].click();", age_btn)
        time.sleep(4)
    except: pass

    # 展開計畫內容
    try:
        expand_btn = driver.find_element(By.CSS_SELECTOR, 'button.js-expand-project')
        driver.execute_script("arguments[0].click();", expand_btn)
    except: pass

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    project_id = project_url.split('/')[-1].split('?')[0]

    pct_tag = soup.find(class_='js-percentage-raised')
    if not pct_tag: return "ERROR"

    time_left_tag = soup.find(class_='js-time-left')
    if time_left_tag and any(x in time_left_tag.text for x in ['天', '時', '分']):
        return "ACTIVE"

    pct_val = int(re.sub(r'[^\d]', '', pct_tag.text)) if re.sub(r'[^\d]', '', pct_tag.text).isdigit() else 0
    status_name = '成功' if pct_val >= 100 else '失敗'

    content_div = soup.find('div', id='project_content')
    content_text = content_div.get_text(separator='\n', strip=True) if content_div else "無法抓取內文"

    price_tags = soup.find_all('div', class_='text-black font-bold text-xl flex items-center')
    plan_prices = [re.search(r'NT\$\s*([\d,]+)', t.text).group(1).replace(',', '') for t in price_tags if re.search(r'NT\$\s*([\d,]+)', t.text)]
    prices_str = " | ".join(plan_prices) if plan_prices else "無價格資訊"

    # === [新增區塊] 計算實際募資天數 ===
    actual_days = 30 # 預設安全值

    try:
        # 嘖嘖的募資期間通常寫在一個 class 包含 "mb-2 text-xs" 或類似的區塊中
        # 格式通常是 "2023/01/01 12:00 – 2023/02/01 12:00"
        date_text_div = soup.find(string=re.compile(r'\d{4}/\d{2}/\d{2}.+–.+\d{4}/\d{2}/\d{2}'))

        if date_text_div:
            # 用正則表達式把兩個日期時間抓出來
            dates = re.findall(r'\d{4}/\d{2}/\d{2} \d{2}:\d{2}', date_text_div)
            if len(dates) == 2:
                # 轉換為 datetime 物件來計算時間差
                from datetime import datetime
                start_date = datetime.strptime(dates[0], "%Y/%m/%d %H:%M")
                end_date = datetime.strptime(dates[1], "%Y/%m/%d %H:%M")

                # 計算相差天數，如果算出來是 0 天(例如當天結束)，至少算 1 天避免除以零
                delta_days = (end_date - start_date).days
                actual_days = max(1, delta_days)
    except Exception as e:
        print(f"天數計算失敗，使用預設值 30 天: {e}")
        actual_days = 30

    return {
        "status": status_name, "project_id": project_id,
        "content_text": content_text, "actual_days": actual_days, "csv_row": [main_name, sub_name, status_name, project_id, prices_str, len(plan_prices)]
    }


# ==========================================
# 4. 圖片OCR
# ==========================================


def batch_easyocr_to_text(image_folder, ocr_output_filepath):
    reader = easyocr.Reader(['ch_tra', 'en'], gpu=True)

    all_extracted_text = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp') # 加上 .webp 防呆

    if not os.path.exists(image_folder):
        print(f"❌ 找不到圖片資料夾：{image_folder}")
        return

    for filename in sorted(os.listdir(image_folder)):
        if filename.lower().endswith(valid_extensions):
            img_path = os.path.join(image_folder, filename)

            try:
                # 【關鍵修正】：不要直接給檔案路徑，而是自己先讀取並強制轉成 RGB
                # 這能過濾掉所有 2通道、4通道(RGBA) 或 CMYK 等奇怪的網頁圖片格式
                img = Image.open(img_path).convert('RGB')

                # 將 PIL 圖片轉成 EasyOCR 看得懂的 numpy 陣列 (矩陣)
                img_np = np.array(img)

                # 把標準化後的陣列餵給 EasyOCR
                result = reader.readtext(img_np, detail=0)

                if result:
                    text = " ".join(result)
                    all_extracted_text.append(text)

            except Exception as e:
                # 如果這張圖片真的壞到讀不出來，就跳過它，不要讓整個爬蟲當機！
                print(f"  ⚠️ 圖片 {filename} 讀取或辨識失敗跳過: {e}")
                continue

    # 如果有抓到文字，就將文字獨立儲存到新的 TXT 檔案中
    if all_extracted_text:
        final_text = "\n".join(all_extracted_text)

        with open(ocr_output_filepath, 'w', encoding='utf-8') as f:
            f.write(final_text)

        filename_only = os.path.basename(ocr_output_filepath)
    else:
        print(f"\n⚠️ 圖片中未辨識到任何有效文字。")



# ==========================================
# 5. 主爬蟲流程 (含崩潰自動重啟機制)
# ==========================================
def main_crawler(target_per_status=16):
    create_directories()
    driver, display = get_driver()

    csv_filename = os.path.join(BASE_DIR, 'projects_summary.csv')
    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['主分類', '次分類', '募資狀態', '專案編號', '方案價格列表', '折扣層數', 'FAQ總題數', 'FAQ更新頻率'])

        for main_name, main_type in MAIN_TYPES.items():
            for sub_name, sub_cat in SUB_CATEGORIES.items():
                success_count, failed_count, page_num = 0, 0, 1

                while success_count < target_per_status or failed_count < target_per_status:
                    print(f"\n分類: {sub_name} 第 {page_num} 頁...")
                    list_url = f"https://www.zeczec.com/categories?category={sub_cat}&type={main_type}&page={page_num}"

                    try:
                        driver.get(list_url)
                    except:
                        print("瀏覽器連線中斷，正在重啟...")
                        driver.quit(); display.stop()
                        driver, display = get_driver()
                        continue

                    time.sleep(3)
                    soup = BeautifulSoup(driver.page_source, 'html.parser')
                    cards = soup.select('a[href^="/projects/"]')
                    project_urls = list(dict.fromkeys([c.get('href') for c in cards if 'comments' not in c.get('href') and 'updates' not in c.get('href')]))

                    if not project_urls: break

                    for url in project_urls:
                        if success_count >= target_per_status and failed_count >= target_per_status: break

                        full_url = "https://www.zeczec.com" + url

                        # 加入 Try-Except 避免單個專案報錯導致整個程式中斷 (解決 ConnectionRefused)
                        try:
                            result = scrape_project_details(driver, full_url, main_name, sub_name)
                        except Exception as e:
                            print(f"專案解析失敗，嘗試重啟瀏覽器: {e}")
                            driver.quit(); display.stop()
                            driver, display = get_driver()
                            continue

                        if result in ["ACTIVE", "ERROR"]: continue
                        status = result["status"]

                        if (status == '成功' and success_count >= target_per_status) or \
                           (status == '失敗' and failed_count >= target_per_status): continue

                        # 建立路徑並存檔
                        # project_folder = os.path.join(BASE_DIR, main_name, sub_name, status, result["project_id"])
                        # os.makedirs(project_folder, exist_ok=True)

                        # with open(os.path.join(project_folder, 'content.txt'), 'w', encoding='utf-8') as f:
                            # f.write(result["content_text"])
                        # writer.writerow(result["csv_row"])

                        # 執行你的下載任務
                        # zeczec_multimodal_downloader(full_url, result["project_id"], driver, project_folder)

                        status = result["status"]
                        project_slug = result["project_id"]

                        # 1. 產生新編號 (例如 GF1)
                        proj_code = generate_project_code(sub_name, status)
                        full_folder_name = f"{proj_code}_{project_slug}"

                        # 2. 建立專案資料夾
                        project_folder = os.path.join(BASE_DIR, main_name, sub_name, status, full_folder_name)
                        os.makedirs(project_folder, exist_ok=True)

                        # 3. 儲存網頁本身的 content.txt
                        content_filename = f"{proj_code}_content.txt"
                        content_filepath = os.path.join(project_folder, content_filename)
                        with open(content_filepath, 'w', encoding='utf-8') as f:
                            f.write(result["content_text"])

                        # 4. 抓取 FAQ
                        funding_duration = result.get("actual_days", 30)
                        faq_count, faq_freq = process_faq_data(driver, full_url, proj_code, project_folder, funding_days=funding_duration)
                        print(f"  ↳ [進度] 總天數: {funding_duration} 天 | 問答總數: {faq_count} 題 | 計算出更新頻率: {faq_freq}")

                        # 5. 更新並寫入 CSV
                        updated_csv_row = result["csv_row"]
                        updated_csv_row[3] = proj_code
                        updated_csv_row.extend([faq_count, faq_freq])
                        writer.writerow(updated_csv_row)

                        # 6. 執行下載任務 (先下載圖片，OCR 才會有東西掃描)
                        zeczec_multimodal_downloader(full_url, project_slug, driver, project_folder, proj_code)

                        # 7. 執行 OCR 辨識，並指定獨立的新檔案名稱
                        img_folder_path = os.path.join(project_folder, f"{proj_code}_images")
                        ocr_filename = f"{proj_code}_ocr_content.txt"
                        ocr_filepath = os.path.join(project_folder, ocr_filename)
                        batch_easyocr_to_text(img_folder_path, ocr_filepath)
                        # --- [修改區塊結束] ---

                        if status == '成功': success_count += 1
                        else: failed_count += 1
                        print(f"完成：{project_slug} ({status})")

                    page_num += 1

    driver.quit()
    display.stop()
    print("\n所有採集任務已順利結束！")

#執行

if __name__ == "__main__":
    main_crawler(target_per_status=16)

    # print("\n正在將雲端的資料夾壓縮成ZIP檔，請稍候...")
    # source_dir = '/content/drive/MyDrive/ZecZec_Group_Data'
    # output_filename = '/content/drive/MyDrive/ZecZec_Dataset_Backup'

    # # 執行壓縮
    # shutil.make_archive(output_filename, 'zip', source_dir)
    # print(f"壓縮成功！ Google雲端硬碟查看{output_filename}.zip")


Mounted at /content/drive
資料夾結構建立完成！
啟動Chrome...

分類: 教育 第 1 頁...
  ↳ [進度] 總天數: 59 天 | 問答總數: 5 題 | 計算出更新頻率: 0.0847
 [影音任務] 正在掃描：rogerems
正在掃描圖片...
圖片下載完成：共 21 張
下載影片: https://www.youtube.com/watch?v=uuJeKGOqi24


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete完成：rogerems (成功)
  ↳ [進度] 總天數: 50 天 | 問答總數: 0 題 | 計算出更新頻率: 0.0
 [影音任務] 正在掃描：frc6998-2026fundraisingproject
正在掃描圖片...
圖片下載完成：共 13 張
下載影片: https://www.youtube.com/watch?v=zmCfppxBAjA
完成：frc6998-2026fundraisingproject (成功)
  ↳ [進度] 總天數: 77 天 | 問答總數: 0 題 | 計算出更新頻率: 0.0
 [影音任務] 正在掃描：nthuracing2025-26
正在掃描圖片...
圖片下載完成：共 24 張
完成：nthuracing2025-26 (成功)
  ↳ [進度] 總天數: 106 天 | 問答總數: 0 題 | 計算出更新頻率: 0.0
 [影音任務] 正在掃描：en-kuang-afi
正在掃描圖片...
圖片下載完成：共 17 張
下載影片: https://www.youtube.com/watch?v=oHhgjhSNvcg
完成：en-kuang-afi (成功)
  ↳ [進度] 總天數: 52 天 | 問答總數: 4 題 | 計算出更新頻率: 0.0769
 [影音任務] 正在掃描：toyroyal-flex-plus
正在掃描圖片...
圖片下載完成：共 49 張
下載影片: https://www.youtube.com/watch?v=zRMQHS0Envk
下載影片: https://www.youtube.com/watch?v=qojslqdfkNY


ERROR: [youtube] qojslqdfkNY: This video is not available


下載影片: https://www.youtube.com/watch?v=IfqTkqt4SUQ
下載影片: https://www.youtube.com/watch?v=RPEHtFKDtys


...

完成：findyourlove-enginneer (失敗)
  ↳ [進度] 總天數: 41 天 | 問答總數: 5 題 | 計算出更新頻率: 0.122
 [影音任務] 正在掃描：onbalance
正在掃描圖片...
圖片下載完成：共 16 張
完成：onbalance (失敗)
  ↳ [進度] 總天數: 23 天 | 問答總數: 4 題 | 計算出更新頻率: 0.1739
 [影音任務] 正在掃描：zhong-yuan-chan-pin-she-ji-zu-bi-ye-zhan-man-tai-zi-jin-mu-zi
正在掃描圖片...
圖片下載完成：共 13 張
完成：zhong-yuan-chan-pin-she-ji-zu-bi-ye-zhan-man-tai-zi-jin-mu-zi (失敗)

分類: 遊戲 第 1 頁...
  ↳ [進度] 總天數: 18 天 | 問答總數: 5 題 | 計算出更新頻率: 0.2778
 [影音任務] 正在掃描：TESBoSE
正在掃描圖片...
圖片下載完成：共 18 張
完成：TESBoSE (成功)
  ↳ [進度] 總天數: 4 天 | 問答總數: 5 題 | 計算出更新頻率: 1.25
 [影音任務] 正在掃描：FateforgeChroniclesOfKaanKinOfTheWild
正在掃描圖片...
圖片下載完成：共 69 張
完成：FateforgeChroniclesOfKaanKinOfTheWild (成功)
  ↳ [進度] 總天數: 17 天 | 問答總數: 6 題 | 計算出更新頻率: 0.3529
 [影音任務] 正在掃描：SweetLands
正在掃描圖片...
圖片下載完成：共 72 張
完成：SweetLands (成功)
  ↳ [進度] 總天數: 68 天 | 問答總數: 8 題 | 計算出更新頻率: 0.1176
 [影音任務] 正在掃描：harry-potter
正在掃描圖片...
圖片下載完成：共 50 張
完成：harry-potter (成功)
  ↳ [進度] 總天數: 17 天 | 問答總數: 7 題 | 計算出更新頻率: 0.4118
 [影音任務] 正在掃描：ThunderRoadVendetta
正在掃描圖片...
圖片下載完成：共 156 張
完

ERROR: [youtube] dfSPkchccUI: This video is not available


完成：buzzer-beater (失敗)
  ↳ [進度] 總天數: 98 天 | 問答總數: 3 題 | 計算出更新頻率: 0.0306
 [影音任務] 正在掃描：feng-xian-zhi-lu-huan-xiao-sheng-zhong-ren-shi-bao-xian-bing-wan-cheng-ren-sheng-gui-hua
正在掃描圖片...
圖片下載完成：共 9 張
完成：feng-xian-zhi-lu-huan-xiao-sheng-zhong-ren-shi-bao-xian-bing-wan-cheng-ren-sheng-gui-hua (失敗)
  ↳ [進度] 總天數: 49 天 | 問答總數: 0 題 | 計算出更新頻率: 0.0
 [影音任務] 正在掃描：myumyuhome-mahjong
正在掃描圖片...
圖片下載完成：共 24 張
下載影片: https://www.youtube.com/watch?v=H1nJxeEiAKY
完成：myumyuhome-mahjong (失敗)

分類: 遊戲 第 9 頁...
  ↳ [進度] 總天數: 31 天 | 問答總數: 0 題 | 計算出更新頻率: 0.0
 [影音任務] 正在掃描：hong-kong-traditional-culture-card-game-villain-hitting
正在掃描圖片...
圖片下載完成：共 15 張
下載影片: https://www.youtube.com/watch?v=RnyCkOcO8Hw
完成：hong-kong-traditional-culture-card-game-villain-hitting (失敗)
  ↳ [進度] 總天數: 25 天 | 問答總數: 4 題 | 計算出更新頻率: 0.16
 [影音任務] 正在掃描：DrLivingstonPuzzles
正在掃描圖片...
圖片下載完成：共 31 張
下載影片: https://www.youtube.com/watch?v=zztn3iKixFI
完成：DrLivingstonPuzzles (失敗)
  ↳ [進度] 總天數: 47 天 | 問答總數: 4 題 | 計算出更新頻率: 0.0851
 [影音任務] 正在掃描：keel-remains
正

ERROR: [youtube] OMUdIAuxhx4: Video unavailable


完成：parent-child-DIY (失敗)
  ↳ [進度] 總天數: 33 天 | 問答總數: 7 題 | 計算出更新頻率: 0.2121
 [影音任務] 正在掃描：tableteam
正在掃描圖片...
圖片下載完成：共 36 張
下載影片: https://www.youtube.com/watch?v=OjetEvwZM4I
下載影片: https://www.youtube.com/watch?v=fFXRZvdJjVI
下載影片: https://www.youtube.com/watch?v=E7zIaYDKacA
完成：tableteam (失敗)

所有採集任務已順利結束！
